In [27]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

dumb_llm = init_chat_model("openai:gpt-3.5-turbo")
average_llm = init_chat_model("openai:gpt-4")
smart_llm = init_chat_model("openai:gpt-5")

In [28]:
class State(TypedDict):

    document: str
    summary: str
    sentiment: str
    key_points: str
    recommendations: str
    final_analysis: str


In [29]:
def get_summary(state: State):
    response = llm.invoke(f"write a 3-sentence summary of the following document: {state['document']}")
    return{
        "summary": response.content,
    }

def get_sentiment(state: State):
    response = llm.invoke(f"Analyze the sentiment and tone of the following document: {state['document']}")
    return{
        "sentiment": response.content,
    }

def get_key_points(state: State):
    response = llm.invoke(f"List the 5 most important points from the following document: {state['document']}")
    return{
        "key_points": response.content,
    }

def get_recommendations(state: State):
    response = llm.invoke(f"Based on the document, list 3 recommended next steps: {state['document']}")
    return{
        "recommendations": response.content,
    }

def get_final_analysis(state: State):
    response = llm.invoke(f"""
    Give me an analysis of the following report

    DOCUMENT ANALYSIS REPORT
    ========================

    EXECUTIVE SUMMARY:
    {state['summary']}

    SENTIMENT ANALYSIS:
    {state['sentiment']}

    KEY POINTS:
    {state.get("key_points", "")}

    RECOMMENDATIONS:
    {state.get('recommendation', "N/A")}
    """)
    return{
        "final_analysis": response.content,
    }

In [30]:
graph_builder = StateGraph(State)

graph_builder.add_node("get_summary", get_summary)
graph_builder.add_node("get_sentiment", get_sentiment)
graph_builder.add_node("get_key_points", get_key_points)
graph_builder.add_node("get_recommendations", get_recommendations)
graph_builder.add_node("get_final_analysis", get_final_analysis)

graph_builder.add_edge(START, "get_summary")
graph_builder.add_edge(START, "get_sentiment")
graph_builder.add_edge(START, "get_key_points")
graph_builder.add_edge(START, "get_recommendations")

graph_builder.add_edge("get_summary", "get_final_analysis")
graph_builder.add_edge("get_sentiment", "get_final_analysis")
graph_builder.add_edge("get_key_points", "get_final_analysis")
graph_builder.add_edge("get_recommendations", "get_final_analysis")
graph_builder.add_edge("get_final_analysis", END)


graph = graph_builder.compile()

In [31]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()

for chunk in graph.stream(
    {"document": document},
    stream_mode="updates",
    ):
    print(chunk, "\n")
    

{'get_summary': {'summary': 'In response to a slowdown in job gains and rising downside risks to employment, alongside elevated inflation, the Federal Open Market Committee has decided to lower interest rates by a quarter of a percentage point while continuing to reduce securities holdings. Economic growth has moderated, with the GDP rising at 1.5% in the first half of the year and consumer spending slowing, despite a pickup in business investment. The unemployment rate slightly increased to 4.3%, and though the labor market shows signs of softening, inflation remains above the 2% target, prompting a recalibration of monetary policy to balance the dual mandate of maximum employment and stable prices.'}} 

{'get_recommendations': {'recommendations': "Based on the detailed document, here are three recommended next steps for the Federal Reserve:\n\n1. **Continue Monitoring Economic Indicators:** The Federal Reserve should maintain vigilant observation of economic data, particularly focusi